In [14]:
# Устанавливаем библиотеки (выполняется ~1-2 мин)
!pip install pdfplumber spacy scikit-learn sentence-transformers
!python -m spacy download ru_core_news_sm

# Отключаем шумные прогресс-бары HuggingFace (опционально, но удобно в Colab)
import re
import os
import json
import numpy as np
from collections import defaultdict, Counter
from typing import List, Dict, Any, Optional, Tuple
import spacy
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import pdfplumber
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 29.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [17]:
from google.colab import files
import os

uploaded = files.upload()  # откроется диалог выбора PDF
pdf_dir = "/content/uploaded_pdfs"
os.makedirs(pdf_dir, exist_ok=True)
for filename in uploaded.keys():
    with open(os.path.join(pdf_dir, filename), 'wb') as f:
        f.write(uploaded[filename])
print(f"✅ Загружено файлов: {len(uploaded)}")

✅ Загружено файлов: 0


In [34]:
class BulletSourceAligner:
    """
    Выравнивает буллиты с исходными новостями из нумерованного списка.
    Определяет, какие новости «послужили источником» для каждого буллита.
    """
    def __init__(self, embedding_model: str = "sentence-transformers/LaBSE"):
        self.emb_model = SentenceTransformer(embedding_model)

    def _text_similarity(self, text1: str, text2: str) -> float:
        """Косинусная близость эмбеддингов"""
        e1 = self.emb_model.encode([text1], show_progress_bar=False)[0]
        e2 = self.emb_model.encode([text2], show_progress_bar=False)[0]
        return cosine_similarity([e1], [e2])[0][0]

    def align_bullet_to_sources(self, bullet: str, news_items: List[Dict[str, str]],
                                threshold: float = 0.65) -> List[Dict[str, str]]:
        """
        Находит новости, которые вероятнее всего послужили источником для буллита.
        Возвращает список совпадений с оценкой релевантности.
        """
        matches = []
        for news in news_items:
            # Сравниваем буллит с заголовком новости
            score = self._text_similarity(bullet, news['title'])
            # Бонус за совпадение ключевых сущностей (упрощённо: по числам и %%)
            bullet_nums = set(re.findall(r'[\d,]+%|[\d,.]+\s*(млн|млрд|тыс|б/с|долл|руб|т)', bullet, re.I))
            news_nums = set(re.findall(r'[\d,]+%|[\d,.]+\s*(млн|млрд|тыс|б/с|долл|руб|т)', news['title'], re.I))
            if bullet_nums & news_nums:
                score += 0.15
            if score >= threshold:
                matches.append({'news': news, 'score': round(score, 3)})
        return sorted(matches, key=lambda x: -x['score'])


class DynamicCriteriaRefiner:
    """
    Генератор конкретных, data-driven критериев отбора буллитов.
    Адаптирован под малые выборки и прямое извлечение паттернов из утверждённых буллитов.
    """
    def __init__(self, emb_model: str = "sentence-transformers/LaBSE",
                 nlp_model: str = "ru_core_news_sm"):
        self.nlp = spacy.load(nlp_model)
        self.emb = SentenceTransformer(emb_model)

        # Хранилища паттернов
        self.bullet_patterns = defaultdict(lambda: Counter())  # section -> {feature: count}
        self.news_patterns = defaultdict(lambda: Counter())    # section -> {feature: count}
        self.section_bullets = defaultdict(list)
        self.section_news = defaultdict(list)

        # Динамические экстракторы паттернов (без жёсткого списка фич)
        self.extractors = [
            self._extract_quantitative_structure,
            self._extract_domain_entities,
            self._extract_action_signals,
            self._extract_temporal_horizon,
            self._extract_source_type,
            self._extract_topic_cluster_keywords,
        ]

    # ==================== ПАРСИНГ ====================

    def parse_pdf(self, path: str, debug: bool = False) -> Dict[str, Dict]:
        """Парсит PDF с учётом \uf0b7 и многострочных буллитов"""
        with pdfplumber.open(path) as pdf:
            text = "\n".join(p.extract_text() or "" for p in pdf.pages)

        # Очистка артефактов
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
        text = re.sub(r'\s{3,}', ' ', text)
        lines = [l.strip() for l in text.split('\n') if l.strip()]

        if debug:
            print(f"\n🔍 DEBUG {os.path.basename(path)}: строк={len(lines)}")
            for i, l in enumerate(lines[:35]):
                print(f"{i:2d}: {repr(l[:70])}")

        sections = {}
        SECTION_NAMES = [
            "I. Мировая экономика и сырьевые рынки",
            "II. Российская экономика",
            "III. Инфляция"
        ]

        # Все возможные маркеры буллитов
        BULLET_MARKERS = ['•', '‣', '◦', '▪', '●', '-', '–', '—', '\uf0b7', '\u2022']
        BULLET_PATTERN = '^[' + re.escape(''.join(BULLET_MARKERS)) + r']\s*'

        current_section = None
        bullets, news = [], []
        state = 'await_section'

        i = 0
        while i < len(lines):
            line = lines[i]

            # 1. Детект раздела
            if any(line == sec for sec in SECTION_NAMES):
                if current_section and (bullets or news):
                    sections[current_section] = {'bullets': bullets, 'news': news}
                current_section = line
                bullets, news = [], []
                state = 'in_bullets'
                i += 1
                continue

            if current_section:
                if state == 'in_bullets':
                    # Буллит: собираем многострочные
                    if re.match(BULLET_PATTERN, line):
                        parts = [re.sub(BULLET_PATTERN, '', line).strip()]
                        i += 1
                        while i < len(lines):
                            nxt = lines[i]
                            if re.match(BULLET_PATTERN, nxt): break
                            if re.match(r'^\d+\.\s+.+', nxt): break
                            if any(nxt == s for s in SECTION_NAMES): break
                            if nxt.strip() and len(nxt.strip()) > 5:
                                parts.append(nxt.strip())
                            i += 1
                        full = ' '.join(parts)
                        if len(full) > 30:
                            bullets.append(full)
                            if debug: print(f"   🔸 Буллит: {full[:60]}...")
                        continue

                    # Переход к новостям
                    if re.match(r'^1\.\s+.+', line):
                        state = 'in_news'
                        if debug: print("   → Новости")

                if state == 'in_news':
                    m = re.match(r'^(\d+)\.\s+(.+)', line)
                    if m:
                        idx, title = int(m.group(1)), m.group(2).strip()
                        url = ''
                        if 'http' in line:
                            um = re.search(r'(https?://[^\s]+)', line)
                            if um: url = um.group(1)
                        elif i+1 < len(lines) and lines[i+1].strip().startswith('http'):
                            url = lines[i+1].strip()
                            i += 1
                        if title and url:
                            news.append({'idx': idx, 'title': title, 'url': url})
                            if debug: print(f"   📰 #{idx}: {title[:40]}...")
                    i += 1
                    continue
            i += 1

        if current_section and (bullets or news):
            sections[current_section] = {'bullets': bullets, 'news': news}

        if debug:
            tb = sum(len(s['bullets']) for s in sections.values())
            tn = sum(len(s['news']) for s in sections.values())
            print(f"📊 ИТОГО: разделов={len(sections)}, Б={tb}, Н={tn}")

        return sections

    # ==================== ДИНАМИЧЕСКИЕ ЭКСТРАКТОРЫ ====================
    def _extract_quantitative_structure(self, text: str) -> List[str]:
        patterns = []
        if re.search(r'[\d,.]+\s*(%|млн|млрд|тыс|б/с|долл|руб|т|коп)', text): patterns.append("HAS_METRIC")
        if re.search(r'(рост|снижение|увелич|паден|измен)[иея]?\s+на?\s+[\d,.]+', text): patterns.append("HAS_CHANGE")
        if re.search(r'в \d+ раз|в \d+ п\.?п\.?|в \d+ раза', text): patterns.append("HAS_MULTIPLIER")
        return patterns

    def _extract_domain_entities(self, text: str) -> List[str]:
        doc = self.nlp(text)
        entities = []
        for ent in doc.ents:
            if ent.label_ in ['GPE','ORG','PRODUCT'] and len(ent.text) < 25:
                entities.append(f"ENT:{ent.label_}:{ent.text.lower()}")
        return list(set(entities))[:4]

    def _extract_action_signals(self, text: str) -> List[str]:
        signals = []
        t = text.lower()
        if re.search(r'(санкц|пошлин|квот|запрет|льгот|НДС|налог|утильсбор|маркировк)', t): signals.append("REG_ACTION")
        if re.search(r'(экспорт|импорт|поставк|логистик|блок|пролив|танкер)', t): signals.append("TRADE_FLOW")
        if re.search(r'(конфликт|переговор|перемирие|военн|учения|эскалация)', t): signals.append("GEOPOL_SHOCK")
        if re.search(r'(прогноз|ожидает|МВФ|МЭА|ОПЕК|ФРС|ЦБ|Росстат|Минэнерго)', t): signals.append("MACRO_FORECAST")
        if re.search(r'(ипотека|кредит|выдач|одобрен|ставка|ликвидность)', t): signals.append("CREDIT_MARKET")
        return signals

    def _extract_temporal_horizon(self, text: str) -> List[str]:
        if re.search(r'(в \d+ квартал|в \d+ году|к \d+ году|с \d+ \w+|по итогам \d+)', text): return ["TIME_HORIZON"]
        return []

    def _extract_source_type(self, text: str) -> List[str]:
        domain = re.search(r'https?://(?:www\.)?([^/\s]+)', text)
        if domain: return [f"SRC:{domain.group(1)}"]
        return []

    def _extract_topic_cluster_keywords(self, text: str) -> List[str]:
        doc = self.nlp(text)
        return [t.lemma_.lower() for t in doc if t.pos_ in ('NOUN','PROPN') and not t.is_stop and len(t.text) > 3][:3]

    # ==================== ОБУЧЕНИЕ КРИТЕРИЯМ ====================
    def update_from_section(self, sec: str, bullets: List[str], news: List[Dict]):
        self.section_bullets[sec].extend(bullets)
        self.section_news[sec].extend(news)

        # 1. Паттерны из утверждённых буллитов (ground truth)
        for b in bullets:
            for ex in self.extractors:
                for p in ex(b):
                    self.bullet_patterns[sec][p] += 1

        # 2. Паттерны из полного пула новостей (baseline)
        for n in news:
            for ex in self.extractors:
                for p in ex(n['title']):
                    self.news_patterns[sec][p] += 1

    def _compute_selection_rates(self, sec: str) -> Dict[str, float]:
        """Возвращает вероятность выбора признака: P(признак | выбран в буллит)"""
        rates = {}
        all_feats = set(self.bullet_patterns[sec].keys()) | set(self.news_patterns[sec].keys())
        for f in all_feats:
            sel = self.bullet_patterns[sec].get(f, 0)
            total = sel + self.news_patterns[sec].get(f, 0)
            rates[f] = sel / total if total > 0 else 0.0
        return rates

    def _extract_emergent_themes(self, sec: str) -> List[str]:
        """Извлекает темы с фильтрацией шума"""
        if len(self.section_bullets[sec]) < 3:
            return []

        # Стоп-слова + чёрный список шумных фраз
        ru_stop = {
            'и','в','во','не','что','он','на','я','с','со','как','а','то','все','она',
            'так','его','но','да','ты','к','у','же','вы','за','бы','по','только','ее',
            'мне','было','вот','от','меня','еще','нет','о','из','ему','теперь','когда',
            'даже','ну','вдруг','ли','если','уже','или','ни','быть','был','него','до',
            'вас','нибудь','опять','уж','вам','сказал','ведь','там','потом','себя',
            'ничего','ей','может','они','тут','где','есть','надо','ней','для','мы',
            'тебя','их','чем','была','сам','чтоб','без','будто','человек','чего','раз',
            'тоже','себе','под','жизнь','будет','ж','тогда','кто','этот','говорил',
            'того','потому','этого','какой','совсем','ним','здесь','этом','один','почти',
            'мой','тем','чтобы','нее','кажется','сейчас','были','куда','зачем','сказать',
            'всех','никогда','сегодня','можно','при','наконец','два','об','другой','хоть',
            'после','над','больше','тот','через','эти','нас','про','всего','них','какая',
            'много','разве','сказала','три','эту','моя','впрочем','хорошо','свою','этой',
            'перед','иногда','лучше','чуть','том','нельзя','такой','им','более','всегда',
            'конечно','всю','между','год','лет','п','г','до','на','по','год','млн','млрд','тыс'
        }
        THEME_BLACKLIST = {
            'аналитический комментарий', 'комментарий дип', 'это произошло',
            'это вызвало', 'на фоне', 'в результате', 'по данным', 'согласно',
            'дип', 'новости о состоянии', 'ключевых отраслей'
        }

        vec = TfidfVectorizer(
            ngram_range=(2, 4),
            max_features=15,
            stop_words=list(ru_stop),
            min_df=1
        )
        X = vec.fit_transform(self.section_bullets[sec])
        scores = np.asarray(X.sum(axis=0)).flatten()
        top_idx = scores.argsort()[-5:][::-1]
        themes = [vec.get_feature_names_out()[i] for i in top_idx]

        # Фильтруем шум
        return [t for t in themes if not any(b in t.lower() for b in THEME_BLACKLIST)]

    def _decode_feature(self, f: str) -> str:
        mapping = {
            "HAS_METRIC": "содержит точную количественную оценку",
            "HAS_CHANGE": "указывает динамику изменения (рост/снижение)",
            "HAS_MULTIPLIER": "содержит кратность изменения (в X раз/п.п.)",
            "REG_ACTION": "описывает регуляторное действие (налоги, пошлины, квоты)",
            "TRADE_FLOW": "касается торговых потоков, логистики, экспорта/импорта",
            "GEOPOL_SHOCK": "связано с геополитикой, конфликтами, санкциями",
            "MACRO_FORECAST": "содержит прогноз или данные макроорганизаций",
            "CREDIT_MARKET": "касается кредитования, ипотеки, ставок",
            "TIME_HORIZON": "имеет чёткий временной горизонт"
        }
        if f in mapping: return mapping[f]
        if f.startswith("ENT:"): return f"ключевая сущность: {f.split(':',2)[-1]}"
        if f.startswith("SRC:"): return f"источник: {f.split(':',1)[-1]}"
        return f.replace("_", " ").lower()

    # ==================== ГЕНЕРАЦИЯ ПРОМПТА ====================
    def generate_criteria_prompt(self, sec: str) -> str:
        rates = self._compute_selection_rates(sec)
        themes = self._extract_emergent_themes(sec)

        # Сглаживание Лапласа + порог значимости
        strong_feats = [
            (f, (self.bullet_patterns[sec].get(f, 0) + 1) /
                (self.bullet_patterns[sec].get(f, 0) + self.news_patterns[sec].get(f, 0) + 2))
            for f in set(self.bullet_patterns[sec].keys()) | set(self.news_patterns[sec].keys())
            if self.bullet_patterns[sec].get(f, 0) >= 2  # минимум 2 вхождения в буллитах
        ]
        strong_feats = [(f, r) for f, r in strong_feats if r > 0.4][:6]

        if not strong_feats and not themes:
            return f"⚠️ Недостаточно данных для раздела: {sec}"

        blocks = []

        # 1. Тематические приоритеты (из кластеризации)
        if themes:
            blocks.append("📌 Тематические приоритеты (на основе прошлых выпусков):")
            for t in themes[:3]:
                blocks.append(f"   • {t}")

        # 2. Обязательные/желательные признаки
        if strong_feats:
            blocks.append("✅ Признаки, повышающие шанс отбора:")
            for f, r in strong_feats:
                blocks.append(f"   • {self._decode_feature(f)} (вероятность отбора: {r:.0%})")

        # 3. Конкретные правила форматирования (выводятся из структуры буллитов)
        blocks.append("📝 Формат буллита:")
        blocks.append("   • [Суть] [Цифра/динамика] [Причина/контекст] [Источник/орган]")
        blocks.append("   • Избегать общих фраз без количественной привязки")
        blocks.append("   • Одна новость = один буллит (без агрегации разнородных тем)")

        # 4. Отсечки (что НЕ брать)
        blocks.append("🚫 Исключать:")
        blocks.append("   • Узкоотраслевые события без макро- или ценового эффекта")
        blocks.append("   • Новости без источника или без количественной оценки")
        blocks.append("   • Повторы уже озвученных в предыдущих выпусках трендов без новых цифр")

        return f"""
РАЗДЕЛ: "{sec}"

ЗАДАЧА: Из нумерованного списка выбери 2–3 новости для буллитов.

КРИТЕРИИ ОТБОРА (динамически выведены из анализа утверждённых выпусков):
{chr(10).join(blocks)}

ПРИМЕР УТВЕРЖДЁННОГО ФОРМАТА:
• ФРС США сохранила ставку на уровне 3,5–3,75% (ожидания снижения сдвинуты на июнь) в условиях проинфляционных рисков [Ключевая ставка] [Динамика] [Причина] [Источник]

ВАЖНО: Буллит должен быть строго основан на одной новости из списка. Не добавляй внешние данные.
"""

    def process_batch(self, pdf_dir: str, debug: bool = False):
        import glob
        files = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
        files_to_process = files[:3] if debug else files  # в debug — только 3 файла
        total_b, total_n = 0, 0

        for f in files_to_process:
            try:
                data = self.parse_pdf(f, debug=debug)
                cnt_b = sum(len(c['bullets']) for c in data.values())
                cnt_n = sum(len(c['news']) for c in data.values())
                total_b += cnt_b
                total_n += cnt_n

                for sec, content in data.items():
                    self.update_from_section(sec, content['bullets'], content['news'])

                print(f"✓ {os.path.basename(f)} | Б: {cnt_b}, Н: {cnt_n}")
            except Exception as e:
                print(f"✗ {os.path.basename(f)}: {e}")
                if debug: import traceback; traceback.print_exc()

        print(f"\n📊 ИТОГО: Буллитов={total_b}, Новостей={total_n}")

In [36]:
# 1. Инициализация
refiner = DynamicCriteriaRefiner()

# 2. Сначала отладка на 3 файлах
# refiner.process_batch("/content/uploaded_pdfs", debug=True)

# 3. Если видите "Б: 6, Н: 41" — запускайте на всех 46
refiner.process_batch("/content/uploaded_pdfs", debug=False)

# 4. Вывод критериев
sections = ["I. Мировая экономика и сырьевые рынки",
            "II. Российская экономика",
            "III. Инфляция"]

for s in sections:
    print(f"\n{'='*70}\n📌 {s}\n{'='*70}")
    print(refiner.generate_criteria_prompt(s))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Подборка важный новостей 02-08.08.2025.pdf | Б: 8, Н: 39
✓ Подборка важных новостей 01-05.12.2025.pdf | Б: 7, Н: 37
✓ Подборка важных новостей 05-14.11.2025.pdf | Б: 7, Н: 38
✓ Подборка важных новостей 13-17.10.2025.pdf | Б: 6, Н: 35
✓ Подборка важных новостей 19-23.01.2026.pdf | Б: 7, Н: 32
✓ Подборка важных новостей 20-24.10.2025.pdf | Б: 7, Н: 40
✓ Подборка важных новостей 23-27.03.2026.pdf | Б: 8, Н: 36
✓ Подборка важных новостей 24-28.11.2025.pdf | Б: 7, Н: 38
✓ Подборка важных новостей 30.03-03.04.2026.pdf | Б: 7, Н: 36
✓ Подборка важных новостей 01-05.09.2025.pdf | Б: 8, Н: 34
✓ Подборка важных новостей 02-06.03.2026.pdf | Б: 7, Н: 32
✓ Подборка важных новостей 02-06.06.2025.pdf | Б: 7, Н: 38
✓ Подборка важных новостей 06-10.04.2026.pdf | Б: 7, Н: 37
✓ Подборка важных новостей 06-10.10.2025.pdf | Б: 8, Н: 32
✓ Подборка важных новостей 07-11.07.2025.pdf | Б: 8, Н: 32
✓ Подборка важных новостей 08-12.09.2025.pdf | Б: 7, Н: 37
✓ Подборка важных новостей 08-12.12.2025.pdf 

In [37]:
import json
output = {s: refiner.generate_criteria_prompt(s) for s in sections}
with open("/content/final_criteria.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("✅ Критерии сохранены в final_criteria.json")

✅ Критерии сохранены в final_criteria.json
